In [ ]:
import pandas as pd
from tqdm import tqdm
import json
import sys
from typing import cast
import os
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv

import mlflow
from mlflow.entities import Trace
from mlflow.entities.assessment import Feedback
from loguru import logger

In [27]:
def get_record_based_on_trace(
    data_df: list[dict],
    trace: Trace
):
    """
    Get the corresponding record based on trace's session id.
    Matches trace input preview with dataset record's user_demand.
    """
    trace_input_preview = cast(str, trace.info.request_preview).strip('"').strip("'").replace('\\n', '')

    filtered_records = [
        r for r in data_df
        if r['inputs']['user_demand'].replace('\n', '') == trace_input_preview
    ]

    if not filtered_records:
        logger.warning(f"No matching record for trace input | Preview: {trace_input_preview[:80]}...")
        return None

    logger.debug(f"Found matching record for trace")
    return filtered_records[0]

In [28]:
mlflow.set_tracking_uri("http://100.113.186.28:5000")
mlflow.set_experiment("vds-agent-validation")

<Experiment: artifact_location='mlflow-artifacts:/6', creation_time=1775830300571, experiment_id='6', last_update_time=1775830300571, lifecycle_stage='active', name='vds-agent-validation', tags={}, workspace='default'>

In [22]:
dataset_id = "d-93f708f111864f24aabb4be63a6024b3"
dataset = mlflow.genai.datasets.get_dataset(dataset_id=dataset_id)  # type: ignore
dataset_df = json.loads(dataset.to_json())

In [23]:
traces = cast(pd.DataFrame, mlflow.search_traces(
        locations=['6'],
        filter_string="trace.text LIKE '%I want%'",
        return_type='pandas'
    ))

In [ ]:
import mlflow
from mlflow.entities import Trace, Span, SpanType
from typing import Any, cast
import pandas as pd
from collections import defaultdict

mlflow.set_tracking_uri('http://100.113.186.28:5000')
mlflow.set_experiment('vds-agent-validation')

print('Searching for traces...')
traces_df = cast(pd.DataFrame, mlflow.search_traces(
    locations=['6'],
    filter_string='trace.text LIKE \"%I want%\"',
    return_type='pandas'
))

print(f'Found {len(traces_df)} traces')

all_models = defaultdict(lambda: {
    'input_tokens': 0,
    'output_tokens': 0,
    'total_tokens': 0,
    'call_count': 0,
    'trace_count': 0,
})

for trace_id in traces_df['trace_id']:
    trace = mlflow.get_trace(trace_id)
    if trace is None:
        continue

    models_in_trace = set()

    for span in trace.data.spans:
        if span.span_type == SpanType.LLM or 'llm' in span.name.lower():
            attrs = span.attributes or {}

            model_id = attrs.get('model_id', attrs.get('model', attrs.get('model_name', '')))
            provider = attrs.get('model_provider', attrs.get('provider', ''))

            if model_id:
                model_key = f'{provider}/{model_id}' if provider else model_id
                models_in_trace.add(model_key)

                input_tokens = attrs.get('input_tokens', attrs.get('prompt_tokens', 0))
                output_tokens = attrs.get('output_tokens', attrs.get('completion_tokens', 0))
                total_tokens = attrs.get('total_tokens', input_tokens + output_tokens)
                cost = attrs.get('cost', attrs.get('total_cost', 0.0))

                all_models[model_key]['input_tokens'] += input_tokens
                all_models[model_key]['output_tokens'] += output_tokens
                all_models[model_key]['total_tokens'] += total_tokens
                all_models[model_key]['call_count'] += 1

    for model_key in models_in_trace:
        all_models[model_key]['trace_count'] += 1

print()
print('=' * 60)
print('ALL MODELS USED IN TRACES')
print('=' * 60)

for model_key, data in sorted(all_models.items(), key=lambda x: x[1]['call_count'], reverse=True):
    print()
    print(f'{model_key}:')
    print(f'  Call count:      {data["call_count"]}')
    print(f'  Trace count:     {data["trace_count"]}')
    print(f'  Input tokens:    {data["input_tokens"]:,}')
    print(f'  Output tokens:   {data["output_tokens"]:,}')
    print(f'  Total tokens:    {data["total_tokens"]:,}')

print()
print('=' * 60)


Searching for traces...
